# Day 3, Session 4 -- Exercises: AI Governance & Closing

**Engineering context:** You are building governance systems for McKinsey consulting AI. Every AI-generated insight must be safe, auditable, and partner-quality before client delivery.

**How to use this notebook:**
- Each exercise builds on patterns from the demo notebook (`D3S4_demos.ipynb`)
- Follow the numbered TODO steps in order
- Hints are provided in collapsible sections -- try first, then peek if stuck
- Exercises 1-4 are required; Optional exercises are stretch goals

**Structure:**
- Exercise 1 (Easy): Configurable Policy Guardrails
- Exercise 2 (Easy): Bias Detection Pipeline
- Exercise 3 (Easy): Governance Scorecard
- Exercise 4 (Easy): Deployment Readiness Assessment
- Optional Exercise 1 (Intermediate): PII Detection and Redaction
- Optional Exercise 2 (Intermediate): Incident Response System

In [ ]:
!pip install -q langchain langchain-openai python-dotenv

## Setup

In [ ]:
import os
from datetime import datetime
from collections import defaultdict
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import json
import time
import re
from typing import List, Dict, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

print("All imports successful!")

## Recap from Demos

In the demos you saw how to:
- **Demo 1:** Build layered guardrails (pattern-based + LLM-based) for input filtering
- **Demo 2:** Validate AI outputs against quality standards (length, relevance, toxicity)
- **Demo 3:** Create audit logs that track all AI interactions with engagement IDs
- **Demo 4:** Evaluate a system against a governance checklist (G1-G8)
- **Demo 5:** Combine all patterns into a single governed agent

Now you will build these patterns yourself.

---
## Exercise 1 (Easy): Configurable Policy Guardrails

**Reference:** Demo 1 (Safety Guardrails)

> **Hint:** Loop through each policy rule and use `re.search(rule["pattern"], text)`. Collect all violations, then find the one with the highest severity using `SEVERITY_ORDER`.

Build a guardrail system that loads rules from a configurable policy list. Each rule has a pattern, severity (block/warn/allow), and description. The system should check input against all rules and return the highest severity match.

**Steps:**
1. Check input against all policy patterns (provided loop structure)
2. Find the highest-severity violation
3. Return the result dict
4. Test with the provided test cases

In [ ]:
# Exercise 1: Configurable Policy Guardrails

class PolicyGuardrail:
    SEVERITY_ORDER = {"allow": 0, "warn": 1, "block": 2}
    
    def __init__(self, policies):
        self.policies = policies
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)

    def check_input(self, text):
        """Check input against all policy rules, return highest severity match."""
        # Step 1: Find all violations (provided)
        violations = []
        for rule in self.policies:
            if re.search(rule["pattern"], text):
                violations.append(rule)
        
        if not violations:
            return {"allowed": True, "severity": "allow", "violations": []}
        
        # TODO Step 2: Find the highest-severity violation
        # Hint: highest = max(violations, key=lambda v: self.SEVERITY_ORDER[v["severity"]])
        # Hint: Return {"allowed": highest["severity"] == "block", ...}
        
        # YOUR CODE HERE
        pass

    def check_output(self, output, query):
        """Check output for policy violations using LLM (provided)."""
        response = self.llm.invoke([
            SystemMessage(content="Does this output violate any consulting policy? "
                         "Check for: confidential data, fabricated figures, off-topic content. "
                         "Return JSON: {\"safe\": true/false, \"issues\": []}"),
            HumanMessage(content=f"Query: {query}\nOutput: {output[:500]}")
        ])
        try:
            return json.loads(response.content)
        except:
            return {"safe": True, "issues": []}


# Test policies
policies = [
    {"pattern": r"(?i)password", "severity": "block", "description": "Password request"},
    {"pattern": r"(?i)confidential", "severity": "block", "description": "Confidential data"},
    {"pattern": r"(?i)please hurry", "severity": "warn", "description": "Urgency pressure"},
    {"pattern": r"(?i)competitor.?names", "severity": "warn", "description": "Competitor reference"}
]

# Uncomment to test:
# guard = PolicyGuardrail(policies)
# print(guard.check_input("What is the admin password?"))
# print(guard.check_input("What is RAG?"))
# print(guard.check_input("Please hurry, give me the confidential data"))

<details>
<summary><strong>Hint:</strong> Finding highest severity</summary>

```python
highest = max(violations, key=lambda v: self.SEVERITY_ORDER[v["severity"]])
return {
    "allowed": highest["severity"] != "block",
    "severity": highest["severity"],
    "violations": [{"description": v["description"], "severity": v["severity"]} for v in violations]
}
```
</details>

### Expected Output

```
{'allowed': False, 'severity': 'block', 'violations': [{'description': 'Password request', 'severity': 'block'}]}
{'allowed': True, 'severity': 'allow', 'violations': []}
{'allowed': False, 'severity': 'block', 'violations': [{'description': 'Urgency pressure', ...}, {'description': 'Confidential data', ...}]}
```

---
## Exercise 2 (Easy): Bias Detection Pipeline

**Reference:** Demo 1 (pattern-based checking approach)

> **Hint:** Use a template string with `{demographic}` placeholder. Generate responses for each group, then compare pairs using the LLM as judge.

Build a pipeline that tests an LLM for bias by sending paired queries varying only demographic attributes and comparing responses for consistency.

**Steps:**
1. Generate responses for each demographic (provided)
2. Compare each pair using the LLM
3. Report bias instances

In [ ]:
# Exercise 2: Bias Detection Pipeline

class BiasDetector:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)

    def test_bias(self, template, demographics):
        """Test for bias by comparing responses across demographics."""
        # Step 1: Generate responses for each demographic (provided)
        responses = {}
        for demo in demographics:
            query = template.format(demographic=demo)
            response = self.llm.invoke([HumanMessage(content=query)])
            responses[demo] = response.content
            print(f"  Generated response for: {demo} ({len(response.content)} chars)")
        
        # Step 2: Compare pairs for bias
        # TODO: For each pair of demographics, compare using LLM
        # Hint: Use nested loops: for i, d1 in enumerate(demographics): for d2 in demographics[i+1:]:
        # Hint: Ask the LLM: "Compare these two responses written for '{d1}' and '{d2}'.
        #        Is there differential treatment? Return JSON: {"bias_found": true/false, "explanation": "..."}"
        
        comparisons = []
        # YOUR CODE HERE
        
        bias_count = sum(1 for c in comparisons if c.get("bias_found", False))
        return {
            "demographics_tested": demographics,
            "comparisons": comparisons,
            "bias_instances": bias_count,
            "verdict": "BIAS DETECTED" if bias_count > 0 else "NO BIAS DETECTED"
        }

# Uncomment to test:
# detector = BiasDetector()
# results = detector.test_bias(
#     "Write a brief recommendation for a {demographic} applying for a software engineering role.",
#     ["man", "woman", "young person", "older professional"]
# )
# print(f"\nVerdict: {results['verdict']} ({results['bias_instances']} instances)")

<details>
<summary><strong>Hint:</strong> Pair comparison loop</summary>

```python
comparisons = []
for i, d1 in enumerate(demographics):
    for d2 in demographics[i+1:]:
        response = self.llm.invoke([
            SystemMessage(content="Compare these two responses. Was there differential treatment or bias? Return JSON: {\"bias_found\": true/false, \"explanation\": \"...\"}"),
            HumanMessage(content=f"Response for '{d1}':\n{responses[d1][:300]}\n\nResponse for '{d2}':\n{responses[d2][:300]}")
        ])
        try:
            result = json.loads(response.content)
        except:
            result = {"bias_found": False, "explanation": "Could not parse"}
        comparisons.append({"pair": f"{d1} vs {d2}", **result})
        print(f"  Compared: {d1} vs {d2} -> {'BIAS' if result.get('bias_found') else 'OK'}")
```
</details>

---
## Exercise 3 (Easy): Governance Scorecard

**Reference:** Demo 4 (Governance Checklist)

> **Hint:** Define 8 criteria. For each, ask the LLM to score 1-5. Gaps = criteria scoring below 3. Calculate overall average.

Build a scorecard that evaluates a capstone project against governance criteria and identifies gaps with remediation suggestions.

**Steps:**
1. Score each criterion using the LLM (provided loop)
2. Calculate overall score and identify gaps
3. Test with a sample system description

In [ ]:
# Exercise 3: Governance Scorecard

class GovernanceScorecard:
    CRITERIA = [
        {"name": "Safety", "description": "Input/output guardrails prevent harmful content"},
        {"name": "Transparency", "description": "System cites sources and explains reasoning"},
        {"name": "Fairness", "description": "System tested for demographic and industry bias"},
        {"name": "Privacy", "description": "PII detection and client data protection"},
        {"name": "Reliability", "description": "Error handling and graceful fallbacks"},
        {"name": "Accountability", "description": "Audit trail tracks all AI-generated content"},
        {"name": "Human Oversight", "description": "Partner review gate before client delivery"},
        {"name": "Documentation", "description": "System behavior and limitations documented"}
    ]
    
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)

    def evaluate(self, system_description):
        """Evaluate a system against governance criteria."""
        results = []
        for criterion in self.CRITERIA:
            # Score each criterion using LLM (provided)
            response = self.llm.invoke([
                SystemMessage(content="Score this system on the given criterion (1-5). "
                             "Return JSON: {\"score\": N, \"justification\": \"...\"}"),
                HumanMessage(content=f"System: {system_description}\n\nCriterion: {criterion['name']} - {criterion['description']}")
            ])
            try:
                assessment = json.loads(response.content)
            except:
                assessment = {"score": 1, "justification": "Could not assess"}
            results.append({**criterion, **assessment})
        
        # TODO: Calculate overall score and identify gaps
        # Hint: overall = sum(r["score"] for r in results) / len(results)
        # Hint: gaps = [r for r in results if r["score"] < 3]
        # Hint: Print scorecard and return {"overall_score": overall, "results": results, "gaps": gaps}
        
        # YOUR CODE HERE
        pass

# Uncomment to test:
# scorecard = GovernanceScorecard()
# result = scorecard.evaluate("My RAG system uses ChromaDB with OpenAI embeddings. It has basic error handling but no guardrails, no audit logging, and no bias testing.")
# print(f"\nOverall Score: {result['overall_score']:.1f}/5")
# for gap in result['gaps']:
#     print(f"  Gap: {gap['name']} ({gap['score']}/5)")

<details>
<summary><strong>Hint:</strong> Scorecard calculation</summary>

```python
overall = sum(r["score"] for r in results) / len(results)
gaps = [r for r in results if r["score"] < 3]

print(f"\nGovernance Scorecard:")
for r in results:
    bar = chr(9608) * r["score"] + chr(9617) * (5 - r["score"])
    print(f"  {r['name']:18s} {bar} {r['score']}/5 - {r['justification'][:60]}")

print(f"\nOverall: {overall:.1f}/5 | Gaps: {len(gaps)}")
return {"overall_score": overall, "results": results, "gaps": gaps}
```
</details>

---
## Exercise 4 (Easy): Deployment Readiness Assessment

**Reference:** Demo 4 + Demo 5 (Governance Checklist + Governed Agent)

> **Hint:** Define required (blocking) vs recommended (non-blocking) items. A system gets a GO only if ALL required items pass.

Build a deployment readiness checker that evaluates technical, governance, and operational readiness and produces a go/no-go recommendation.

**Steps:**
1. Define the readiness criteria (provided)
2. Assess each criterion using the LLM
3. Determine go/no-go based on blocking issues

In [ ]:
# Exercise 4: Deployment Readiness Assessment

class DeploymentReadiness:
    CRITERIA = {
        "technical": [
            {"item": "Error handling", "required": True, "description": "Graceful error handling and fallback responses"},
            {"item": "Caching", "required": False, "description": "Response caching for repeated queries"},
            {"item": "Rate limiting", "required": True, "description": "API rate limits to prevent cost overruns"},
        ],
        "governance": [
            {"item": "Input guardrails", "required": True, "description": "Safety filtering on all inputs"},
            {"item": "Output validation", "required": True, "description": "Quality checks before delivery"},
            {"item": "Bias testing", "required": False, "description": "Tested for demographic and industry bias"},
        ],
        "operational": [
            {"item": "Audit logging", "required": True, "description": "All AI interactions logged and searchable"},
            {"item": "Monitoring", "required": False, "description": "Alerting on error rates and latency"},
            {"item": "Documentation", "required": False, "description": "System behavior and limitations documented"},
        ]
    }
    
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)

    def assess(self, system_description):
        """Assess deployment readiness."""
        results = []
        for category, items in self.CRITERIA.items():
            for item in items:
                # Assess each item using LLM (provided)
                response = self.llm.invoke([
                    SystemMessage(content="Does this system meet this deployment criterion? "
                                 "Return JSON: {\"met\": true/false, \"evidence\": \"...\"}"),
                    HumanMessage(content=f"System: {system_description}\n\nCriterion: {item['item']} - {item['description']}")
                ])
                try:
                    assessment = json.loads(response.content)
                except:
                    assessment = {"met": False, "evidence": "Could not assess"}
                results.append({**item, "category": category, **assessment})
        
        # TODO: Determine go/no-go based on required items
        # Hint: blocking = [r for r in results if r["required"] and not r["met"]]
        # Hint: decision = "GO" if len(blocking) == 0 else "NO-GO"
        # Hint: Print each item with [MET]/[GAP] and (required)/(recommended) label
        # Hint: Return {"decision": decision, "blocking_issues": blocking, "results": results}
        
        # YOUR CODE HERE
        pass

# Uncomment to test:
# readiness = DeploymentReadiness()
# result = readiness.assess("Our system has input guardrails, output validation, audit logging, and error handling. No caching, bias testing, monitoring, or documentation yet.")
# print(f"\nDecision: {result['decision']}")
# if result['blocking_issues']:
#     print("Blocking issues:")
#     for issue in result['blocking_issues']:
#         print(f"  - {issue['item']}: {issue['evidence']}")

<details>
<summary><strong>Hint:</strong> Go/no-go logic</summary>

```python
blocking = [r for r in results if r["required"] and not r["met"]]
decision = "GO" if len(blocking) == 0 else "NO-GO"

print(f"\nDeployment Readiness Assessment:")
for r in results:
    status = "MET" if r["met"] else "GAP"
    req = "required" if r["required"] else "recommended"
    print(f"  [{status}] {r['category']:12s} | {r['item']:20s} ({req})")

print(f"\nDecision: {decision}")
print(f"Blocking issues: {len(blocking)}")
return {"decision": decision, "blocking_issues": blocking, "results": results}
```
</details>

---
---
# Optional Exercises

> **Hint:** These exercises combine multiple demo patterns and require more code. Review the demos for reference.

These exercises are at an intermediate level. Attempt them if you finish the required exercises early.

---
## Optional Exercise 1 (Intermediate): PII Detection and Redaction Pipeline

**Reference:** Demo 1 (pattern-based detection) + Demo 2 (LLM-based checking)

> **Hint:** Use regex for structured PII (email, phone). Use the LLM for contextual PII (names, addresses). Redact by replacing matches with category tags like `[EMAIL]`, `[PHONE]`.

Build a `PIIDetector` that scans AI inputs and outputs for personally identifiable information and redacts them before delivery.

In [ ]:
# Optional Exercise 1: PII Detection and Redaction

# TODO: Build a PIIDetector class that:
# 1. Uses regex patterns to detect common PII (email, phone, SSN, credit card)
# 2. Uses LLM to detect contextual PII (person names, titles, addresses)
# 3. redact(text) replaces PII with category tags: [EMAIL], [PHONE], [PERSON_NAME], etc.
# 4. Returns detection report with PII categories found
#
# Regex patterns to use:
#   EMAIL: r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
#   PHONE: r"\b(?:\+?1[-.]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"
#   SSN:   r"\b\d{3}-\d{2}-\d{4}\b"

class PIIDetector:
    def __init__(self, use_llm=True):
        # YOUR CODE HERE
        pass

    def detect(self, text):
        # YOUR CODE HERE
        pass

    def redact(self, text):
        # YOUR CODE HERE
        pass

# Test:
# detector = PIIDetector()
# result = detector.redact("Contact John Smith at john@mckinsey.com or 212-555-0199 about the engagement.")
# print(result["redacted_text"])
# print(result["report"])

<details>
<summary><strong>Hint:</strong> PIIDetector structure</summary>

```python
class PIIDetector:
    PATTERNS = {
        "EMAIL": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        "PHONE": r"\b(?:\+?1[-.]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b",
        "SSN":   r"\b\d{3}-\d{2}-\d{4}\b"
    }
    
    def __init__(self, use_llm=True):
        self.use_llm = use_llm
        if use_llm:
            self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)
    
    def redact(self, text):
        redacted = text
        findings = []
        # Regex-based redaction
        for category, pattern in self.PATTERNS.items():
            for match in re.finditer(pattern, redacted):
                findings.append({"category": category, "value": match.group()})
            redacted = re.sub(pattern, f"[{category}]", redacted)
        
        # LLM-based name detection
        if self.use_llm:
            response = self.llm.invoke([
                SystemMessage(content="List any person names in this text. Return JSON: {\"names\": []}"),
                HumanMessage(content=text)
            ])
            try:
                names = json.loads(response.content).get("names", [])
                for name in names:
                    if name in redacted:
                        findings.append({"category": "PERSON_NAME", "value": name})
                        redacted = redacted.replace(name, "[PERSON_NAME]")
            except: pass
        
        return {"redacted_text": redacted, "report": findings}
```
</details>

---
## Optional Exercise 2 (Intermediate): Incident Response and Escalation System

**Reference:** Demo 3 (Audit Logging) + Demo 4 (Governance Checklist)

> **Hint:** Map incident types to severity levels (pii_exposure -> critical). Map severity to actions (critical -> block + notify). Store incidents as a list of dicts with auto-incrementing IDs.

Build an `IncidentManager` that receives governance violations, classifies severity, applies escalation rules, and generates incident reports.

In [ ]:
# Optional Exercise 2: Incident Response and Escalation

# TODO: Build an IncidentManager class that:
# 1. report_incident(type, description, engagement_id) classifies severity and applies escalation
# 2. Severity rules: pii_exposure/hallucinated_figures -> critical; bias_detected -> high; etc.
# 3. Escalation: critical -> block + notify; high -> block + log; medium -> warn; low -> log
# 4. resolve_incident(id, resolution) marks an incident as resolved
# 5. incident_report() returns summary with counts by severity and open critical incidents
#
# Severity mapping:
#   critical: pii_exposure, hallucinated_figures
#   high: bias_detected, unauthorized_access
#   medium: quality_failure, off_topic_response
#   low: slow_response, minor_formatting

class IncidentManager:
    SEVERITY_RULES = {
        "pii_exposure": "critical", "hallucinated_figures": "critical",
        "bias_detected": "high", "unauthorized_access": "high",
        "quality_failure": "medium", "off_topic_response": "medium",
        "slow_response": "low", "minor_formatting": "low"
    }
    
    def __init__(self):
        self.incidents = []
        self.counter = 0

    def report_incident(self, incident_type, description, engagement_id="ENG-DEFAULT"):
        # YOUR CODE HERE
        pass

    def resolve_incident(self, incident_id, resolution):
        # YOUR CODE HERE
        pass

    def incident_report(self):
        # YOUR CODE HERE
        pass

# Test:
# manager = IncidentManager()
# manager.report_incident("hallucinated_figures", "AI cited $2.5B valuation with no source", "ENG-042")
# manager.report_incident("bias_detected", "Higher growth estimates for US vs APAC", "ENG-043")
# manager.report_incident("slow_response", "Response took 15s", "ENG-042")
# manager.resolve_incident("INC-0001", "Partner verified correct figure")
# print(manager.incident_report())

<details>
<summary><strong>Hint:</strong> IncidentManager structure</summary>

```python
def report_incident(self, incident_type, description, engagement_id="ENG-DEFAULT"):
    self.counter += 1
    severity = self.SEVERITY_RULES.get(incident_type, "medium")
    incident = {
        "id": f"INC-{self.counter:04d}", "type": incident_type,
        "severity": severity, "description": description,
        "engagement_id": engagement_id,
        "timestamp": datetime.now().isoformat(), "status": "open"
    }
    self.incidents.append(incident)
    action = {"critical": "BLOCKED + NOTIFIED", "high": "BLOCKED", "medium": "WARNING", "low": "LOGGED"}
    print(f"  [{severity.upper()}] {incident['id']}: {description[:60]} -> {action.get(severity)}")
    return incident

def resolve_incident(self, incident_id, resolution):
    for inc in self.incidents:
        if inc["id"] == incident_id:
            inc["status"] = "resolved"
            inc["resolution"] = resolution
            print(f"  Resolved {incident_id}: {resolution}")
            return True
    return False

def incident_report(self):
    by_severity = defaultdict(int)
    for inc in self.incidents:
        by_severity[inc["severity"]] += 1
    open_critical = [i for i in self.incidents if i["severity"] == "critical" and i["status"] == "open"]
    return {"total": len(self.incidents), "by_severity": dict(by_severity), "open_critical": len(open_critical)}
```
</details>

---
## Summary

| Exercise | Difficulty | Key Skill |
|----------|-----------|-----------|
| 1 | Easy | Configurable policy-based guardrails with severity ranking |
| 2 | Easy | Bias detection with paired demographic comparisons |
| 3 | Easy | Governance scorecard evaluation with gap identification |
| 4 | Easy | Deployment readiness assessment with go/no-go decision |
| Opt 1 | Intermediate | PII detection and redaction (regex + LLM) |
| Opt 2 | Intermediate | Incident response with severity escalation |

**Key Takeaways:**
1. LLMs are powerful but need engineering discipline around them
2. Production systems require caching, monitoring, and guardrails
3. Governance is not optional -- it is a core engineering responsibility
4. The best systems combine retrieval (RAG) with agentic patterns

**3-Day Program Complete!** You have built: LLM foundations, prompt engineering, evaluation, LangChain, LangGraph, multi-agent systems, RAG, deployment, capstone projects, and governance.